# autoresearch_bench — Component Debugger

Interactive walkthrough of every component in the `autoresearch_bench` package.
Use this notebook to:
- Inspect configs, prompts, and results
- Test code extraction and patch application
- Run mock end-to-end experiments without a vLLM server
- Connect to a live vLLM server for real LLM calls

---

## 1. Setup & Imports

In [1]:
# Install the package in editable mode (run once)
# !pip install -e '.[dev]'

from __future__ import annotations

import asyncio
import json
from pathlib import Path
from unittest.mock import AsyncMock, MagicMock, patch

# Core package imports
from autoresearch_bench.code_utils import apply_edit, extract_code
from autoresearch_bench.config import ExperimentConfig
from autoresearch_bench.llm.client import LLMClient, _backoff
from autoresearch_bench.llm.models import KNOWN_MODELS, resolve_model
from autoresearch_bench.prompts.builder import PromptBuilder, _build_description
from autoresearch_bench.results import RunResult, StepResult, aggregate_results
from autoresearch_bench.runner import Runner, _build_sampler, _config_to_dict
from autoresearch_bench.samplers.iterative_sampler import IterativeSampler
from autoresearch_bench.samplers.random_sampler import RandomSampler

print("All imports OK ✓")

All imports OK ✓


---
## 2. Config Loading

In [2]:
# Load the bundled example config
example_cfg_path = Path("../configs/example.yaml")

if example_cfg_path.exists():
    cfg = ExperimentConfig.from_yaml(example_cfg_path)
    print("Loaded config from", example_cfg_path)
else:
    # Fallback: build from a dict inline
    cfg = ExperimentConfig._from_dict({
        "vllm": {"base_url": "http://localhost:8000/v1", "api_key": "dummy"},
        "models": [{"name": "gpt-oss-120b"}],
        "problems": ["combinatorics/cap_set"],
        "samplers": [{"type": "random", "num_samples": 5}],
        "runs": {"seeds": [42]},
    })
    print("Created inline config")

Loaded config from ../configs/example.yaml


In [3]:
# Inspect all top-level fields
print("vLLM URL:", cfg.vllm.base_url)
print("Models:", [m.name for m in cfg.models])
print("Problems:", cfg.problems)
print("Seeds:", cfg.runs.seeds)
print("Output dir:", cfg.output_dir)
print()
print("LLM params:")
print("  temperature:", cfg.llm_params.temperature)
print("  max_tokens:", cfg.llm_params.max_tokens)
print("  top_p:", cfg.llm_params.top_p)
print()
print("Samplers:")
for s in cfg.samplers:
    print(f"  {s.type}/{s.mode}  num_samples={s.num_samples}  num_steps={s.num_steps}")

vLLM URL: http://localhost:8000/v1
Models: ['gpt-oss-120b']
Problems: ['combinatorics/cap_set']
Seeds: [42, 123, 456]
Output dir: experiments/results

LLM params:
  temperature: 0.8
  max_tokens: 4096
  top_p: 0.95

Samplers:
  random/full_rewrite  num_samples=50  num_steps=10
  iterative/full_rewrite  num_samples=50  num_steps=10


In [4]:
# Modify params interactively
import dataclasses

tweaked = dataclasses.replace(cfg.llm_params, temperature=0.5, max_tokens=1024)
print("Tweaked LLM params:", tweaked)

Tweaked LLM params: LLMParams(temperature=0.5, max_tokens=1024, top_p=0.95)


---
## 3. Model Registry

In [5]:
# Show all registered model aliases
print("KNOWN_MODELS:")
for alias, model_id in KNOWN_MODELS.items():
    print(f"  {alias!r:30s} → {model_id!r}")

KNOWN_MODELS:
  'gpt-oss-120b'                 → 'gpt-oss-120b'
  'qwen3.5-32b'                  → 'Qwen/Qwen3.5-32B'
  'Qwen/Qwen3.5-32B'             → 'Qwen/Qwen3.5-32B'


In [6]:
# Test resolve_model with various inputs
test_names = [
    "gpt-oss-120b",
    "qwen3.5-32b",
    "Qwen/Qwen3.5-32B",
    "my-custom-model",   # passthrough
]

for name in test_names:
    resolved = resolve_model(name)
    marker = "✓" if resolved == name or resolved in KNOWN_MODELS.values() else "?"
    print(f"  {marker} {name!r:35s} → {resolved!r}")

  ✓ 'gpt-oss-120b'                      → 'gpt-oss-120b'
  ✓ 'qwen3.5-32b'                       → 'Qwen/Qwen3.5-32B'
  ✓ 'Qwen/Qwen3.5-32B'                  → 'Qwen/Qwen3.5-32B'
  ✓ 'my-custom-model'                   → 'my-custom-model'


In [7]:
# Add a custom model alias at runtime
KNOWN_MODELS["my-llm"] = "org/MyLLM-7B"
print("After adding custom alias:", resolve_model("my-llm"))
# Clean up
del KNOWN_MODELS["my-llm"]

After adding custom alias: org/MyLLM-7B


---
## 4. Prompt Builder

In [2]:
try:
    from autoresearch_problems import ProblemSpec

    sample_spec = ProblemSpec(
        name="cap_set",
        category="combinatorics",
        description="Find the largest subset of F_3^n with no three elements in arithmetic progression.",
        output_type="list[list[int]]",
        evaluator_code="",
        evaluator_entrypoint="evaluate",
        evaluator_dependencies=[],
        parameters={"n": 6},
        initial_prompt="Improve the cap set solver to find larger subsets.",
        initial_program="def solve(n):\n    return []\n",
        function_name="solve",
    )
    print("ProblemSpec created ✓")
except ImportError:
    print("autoresearch_problems not available — using a MagicMock spec")
    sample_spec = MagicMock()
    sample_spec.name = "cap_set"
    sample_spec.category = "combinatorics"
    sample_spec.description = "Find the largest cap set."
    sample_spec.initial_prompt = "Improve the solver."
    sample_spec.initial_program = "def solve(n):\n    return []\n"

ProblemSpec created ✓


In [3]:
# Build prompts for both modes
current_program = "def solve(n):\n    return [[0] * n]\n"

for mode in ("full_rewrite", "edit"):
    builder = PromptBuilder(mode=mode)
    messages = builder.build(sample_spec, current_program)
    print(f"\n{'='*60}")
    print(f"Mode: {mode}")
    print(f"{'='*60}")
    for msg in messages:
        print(f"\n[{msg['role'].upper()}]")
        print(msg["content"][:400], "..." if len(msg["content"]) > 400 else "")


Mode: full_rewrite

[SYSTEM]
You are an expert programmer tasked with improving an existing solution to an optimization problem.
Your goal is to produce a better-performing implementation.

Rules:
- Return ONLY the complete, runnable Python code inside a single markdown code block (```python ... ```).
- Do NOT include any explanation, prose, or commentary outside the code block.
- The function signature must match the origina ...

[USER]
## Problem

Find the largest subset of F_3^n with no three elements in arithmetic progression.

Improve the cap set solver to find larger subsets.

## Current Program

```python
def solve(n):
    return [[0] * n]

```

## Task

Rewrite the entire program to achieve a better score.
Focus on algorithmic improvements, smarter search strategies, or more efficient data structures.
Return the complete i ...

Mode: edit

[SYSTEM]
You are an expert programmer tasked with improving an existing solution to an optimization problem.
Your goal is to produce a unif

In [10]:
# Inspect _build_description for various spec configurations
from unittest.mock import MagicMock

cases = [
    ("description + prompt", "My description.", "Extra prompt context."),
    ("description only", "My description.", None),
    ("neither", "", None),
]

for label, desc, prompt in cases:
    mock = MagicMock()
    mock.name = "test_problem"
    mock.description = desc
    mock.initial_prompt = prompt
    result = _build_description(mock)
    print(f"  [{label}] → {result!r}")

  [description + prompt] → 'My description.\n\nExtra prompt context.'
  [description only] → 'My description.'
  [neither] → 'test_problem'


---
## 5. Code Extraction

In [14]:
# Test extract_code with various LLM-like response strings
responses = [
    ("full_rewrite", "Good python block",
     "Here's the solution:\n```python\ndef solve(n): return list(range(n))\n```\n"),
    ("full_rewrite", "Generic block",
     "```\ndef solve(n):\n    pass\n```"),
    ("edit", "Diff block",
     "```diff\n@@ -1 +1 @@\n-old\n+new\n```"),
    ("edit", "Fallback to python",
     "```python\ndef solve(n): return []\n```"),
    ("full_rewrite", "No block (returns None)",
     "Just some plain text, no code here."),
    ("full_rewrite", "Empty string", ""),
]

for mode, label, text in responses:
    result = extract_code(text, mode=mode)
    print(f"  [{mode}/{label}]")
    print(f"    → {repr(result)[:80] if result else None}")

No code block found in LLM response (mode=full_rewrite). Raw text preview: Just some plain text, no code here.
No code block found in LLM response (mode=full_rewrite). Raw text preview: 


  [full_rewrite/Good python block]
    → 'def solve(n): return list(range(n))'
  [full_rewrite/Generic block]
    → 'def solve(n):\n    pass'
  [edit/Diff block]
    → '@@ -1 +1 @@\n-old\n+new'
  [edit/Fallback to python]
    → 'def solve(n): return []'
  [full_rewrite/No block (returns None)]
    → None
  [full_rewrite/Empty string]
    → None


In [15]:
# Test apply_edit with a sample unified diff
original = """def solve(n):
    # Naive: return empty set
    return []
"""

patch_str = """@@ -1,3 +1,4 @@
 def solve(n):
-    # Naive: return empty set
-    return []
+    # Improved: return a greedy cap set
+    result = [[0] * n]
+    return result
"""

patched = apply_edit(original, patch_str)
print("Original:")
print(original)
print("Patched:")
print(patched)

Original:
def solve(n):
    # Naive: return empty set
    return []

Patched:
def solve(n):
    # Improved: return a greedy cap set
    result = [[0] * n]
    return result



In [16]:
# Edge case: malformed patch returns original
bad_patch = "this is not a valid diff at all"
result = apply_edit(original, bad_patch)
print("Malformed patch — got original back:", result == original)

Malformed patch — got original back: True


---
## 6. LLM Client (Mock)

In [17]:
# Demonstrate creating an LLMClient (no real server needed here)
import openai

def make_mock_response(content: str):
    """Build a mock openai ChatCompletion response."""
    mock_choice = MagicMock()
    mock_choice.message.content = content
    mock_resp = MagicMock()
    mock_resp.choices = [mock_choice]
    return mock_resp

print("_backoff values for attempts 0–4:")
for i in range(5):
    print(f"  attempt {i}: {_backoff(i, base=1.0, cap=30.0):.3f}s (randomised)")

_backoff values for attempts 0–4:
  attempt 0: 0.846s (randomised)
  attempt 1: 1.697s (randomised)
  attempt 2: 2.212s (randomised)
  attempt 3: 1.093s (randomised)
  attempt 4: 1.539s (randomised)


In [ ]:
# Mock batch_complete with known responses
async def demo_mock_batch():
    mock_responses = [
        "```python\ndef solve(n): return [[0]*n]\n```",
        "```python\ndef solve(n): return list(range(n))\n```",
        "No code here — will extract None",
    ]

    with patch("openai.AsyncOpenAI") as MockOpenAI:
        mock_oai = MagicMock()
        mock_oai.chat.completions.create = AsyncMock(
            side_effect=[make_mock_response(r) for r in mock_responses]
        )
        MockOpenAI.return_value = mock_oai

        client = LLMClient(base_url="http://localhost:8000/v1", api_key="dummy")
        messages_list = [[{"role": "user", "content": f"q{i}"}] for i in range(3)]
        results = await client.batch_complete("gpt-oss-120b", messages_list)

    for i, r in enumerate(results):
        code = extract_code(r) if isinstance(r, str) else None
        print(f"  response {i}: extracted={code!r[:50] if code else None}")

asyncio.run(demo_mock_batch())

---
## 7. LLM Client (Live — requires running vLLM server)

> ⚠️ **This section requires a running vLLM server.** Skip if not available.

In [4]:
VLLM_URL = "http://localhost:8000/v1"
MODEL_NAME = "gpt-oss-120b"  # Change to your served model name

async def test_live_llm():
    try:
        client = LLMClient(base_url=VLLM_URL, api_key="dummy", max_retries=1)
        messages = [{"role": "user", "content": "Say hello in one sentence."}]
        response = await client.complete(
            model=resolve_model(MODEL_NAME),
            messages=messages,
            temperature=0.7,
            max_tokens=64,
        )
        print("Response:", response)
    except Exception as exc:
        print(f"Could not connect to vLLM at {VLLM_URL}: {exc}")
        print("Start a vLLM server to run this cell.")

asyncio.run(test_live_llm())

RuntimeError: asyncio.run() cannot be called from a running event loop

---
## 8. Problem Loading

In [ ]:
try:
    from autoresearch_problems import registry

    all_problems = registry.list_problems()
    print(f"Available problems ({len(all_problems)}):")
    for p in all_problems:
        print(f"  - {p}")
except Exception as exc:
    print(f"Could not list problems: {exc}")

In [ ]:
try:
    spec = registry.load("combinatorics/cap_set")
    print("ProblemSpec fields:")
    print(f"  name:             {spec.name}")
    print(f"  category:         {spec.category}")
    print(f"  maximize:         {spec.maximize}")
    print(f"  known_best_score: {spec.known_best_score}")
    print(f"  timeout_seconds:  {spec.timeout_seconds}")
    print(f"  parameters:       {spec.parameters}")
    print(f"  output_type:      {spec.output_type}")
    print(f"  tags:             {spec.tags}")
    print(f"  function_name:    {spec.function_name}")
    print(f"\ninitial_program (first 200 chars):")
    print((spec.initial_program or "")[:200])
except Exception as exc:
    print(f"Could not load problem: {exc}")

---
## 9. Evaluation Pipeline

In [ ]:
try:
    from autoresearch_problems import execute_and_evaluate

    eval_result = execute_and_evaluate(spec, spec.initial_program)
    print("EvalResult for initial program:")
    print(f"  score:          {eval_result.score}")
    print(f"  valid:          {eval_result.valid}")
    print(f"  error:          {eval_result.error!r}")
    print(f"  execution_time: {eval_result.execution_time:.3f}s")
    print(f"  metrics:        {eval_result.metrics}")
except Exception as exc:
    print(f"Evaluation failed: {exc}")

---
## 10. Random Sampler (Mock End-to-End)

In [5]:
from autoresearch_problems import EvalResult, ProblemSpec

# Create a spec for demo purposes
demo_spec = ProblemSpec(
    name="cap_set",
    category="combinatorics",
    description="Find the largest cap set in F_3^n.",
    output_type="list[list[int]]",
    evaluator_code="",
    evaluator_entrypoint="evaluate",
    evaluator_dependencies=[],
    parameters={"n": 6},
    maximize=True,
    initial_program="def solve(n):\n    return []\n",
)

async def run_mock_random():
    # Mocked LLM responses
    llm_responses = [
        "```python\ndef solve(n):\n    return [[0]*n]\n```",
        "```python\ndef solve(n):\n    return [[1]*n, [0]*n]\n```",
        "```python\ndef solve(n):\n    return [[i]*n for i in range(3)]\n```",
    ]
    eval_results = [
        EvalResult(score=1.0, valid=True, execution_time=0.01),
        EvalResult(score=2.0, valid=True, execution_time=0.01),
        EvalResult(score=3.0, valid=True, execution_time=0.01),
    ]
    initial_eval = EvalResult(score=0.0, valid=True, execution_time=0.01)

    with (
        patch("openai.AsyncOpenAI") as MockOpenAI,
        patch("autoresearch_bench.samplers.random_sampler.execute_and_evaluate", return_value=initial_eval),
        patch("autoresearch_bench.samplers.base.execute_and_evaluate_batch", return_value=eval_results),
    ):
        mock_oai = MagicMock()
        mock_oai.chat.completions.create = AsyncMock(
            side_effect=[make_mock_response(r) for r in llm_responses]
        )
        MockOpenAI.return_value = mock_oai

        client = LLMClient(base_url="http://localhost:8000/v1", api_key="dummy")
        sampler = RandomSampler(
            num_samples=3,
            client=client,
            model="gpt-oss-120b",
            prompt_builder=PromptBuilder(mode="full_rewrite"),
            llm_params={"temperature": 0.8, "max_tokens": 256, "top_p": 0.95},
        )
        result = await sampler.run(spec=demo_spec, seed=42, config_dict={})

    return result

random_result = asyncio.run(run_mock_random())
print("RunResult summary:")
print(json.dumps(random_result.summary(), indent=2))

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
# Inspect individual steps
print(f"Steps ({len(random_result.steps)}):")
for step in random_result.steps:
    print(f"  step={step.step}  score={step.score}  valid={step.valid}  code_len={len(step.generated_code)}")

---
## 11. Iterative Sampler (Mock End-to-End)

In [6]:
step_responses = [
    # Step 1 (samples 0–1): modest improvement
    "```python\ndef solve(n):\n    return [[0]*n, [1]*n]\n```",
    "```python\ndef solve(n):\n    return [[0]*n]\n```",
    # Step 2 (samples 2–3): big improvement
    "```python\ndef solve(n):\n    return [[i]*n for i in range(5)]\n```",
    "```python\ndef solve(n):\n    return [[0]*n, [2]*n]\n```",
]
step_evals = [
    [EvalResult(score=2.0, valid=True, execution_time=0.01), EvalResult(score=1.0, valid=True, execution_time=0.01)],
    [EvalResult(score=5.0, valid=True, execution_time=0.01), EvalResult(score=3.0, valid=True, execution_time=0.01)],
]

step_call_count = 0
resp_call_count = 0

def side_effect_eval_batch(spec, codes, **kwargs):
    global step_call_count
    result = step_evals[min(step_call_count, 1)]
    step_call_count += 1
    return result

async def run_mock_iterative():
    with (
        patch("openai.AsyncOpenAI") as MockOpenAI,
        patch("autoresearch_bench.samplers.iterative_sampler.execute_and_evaluate",
              return_value=EvalResult(score=0.0, valid=True, execution_time=0.01)),
        patch("autoresearch_bench.samplers.base.execute_and_evaluate_batch",
              side_effect=side_effect_eval_batch),
    ):
        mock_oai = MagicMock()
        mock_oai.chat.completions.create = AsyncMock(
            side_effect=[make_mock_response(r) for r in step_responses]
        )
        MockOpenAI.return_value = mock_oai

        client = LLMClient(base_url="http://localhost:8000/v1", api_key="dummy")
        sampler = IterativeSampler(
            num_steps=2,
            samples_per_step=2,
            client=client,
            model="gpt-oss-120b",
            prompt_builder=PromptBuilder(mode="full_rewrite"),
            llm_params={"temperature": 0.8, "max_tokens": 256, "top_p": 0.95},
        )
        return await sampler.run(spec=demo_spec, seed=42, config_dict={})

iterative_result = asyncio.run(run_mock_iterative())
print("Iterative RunResult summary:")
print(json.dumps(iterative_result.summary(), indent=2))

RuntimeError: asyncio.run() cannot be called from a running event loop

In [ ]:
# Show how best_code evolves across steps
print(f"Initial score: {iterative_result.initial_score}")
print(f"Final best score: {iterative_result.best_score}")
print()
print("Step scores:")
for step in iterative_result.steps:
    print(f"  step={step.step}  score={step.score}  valid={step.valid}")

---
## 12. Results Inspection

In [ ]:
import tempfile

# Create a few RunResult objects
def _make_run_result(sampler_type, seed, best_score):
    return RunResult(
        sampler_type=sampler_type,
        sampler_mode="full_rewrite",
        model="gpt-oss-120b",
        problem="combinatorics/cap_set",
        seed=seed,
        steps=[],
        best_score=best_score,
        best_code="def solve(n): return []",
        initial_score=0.0,
        config_dict={},
        timestamp="2024-01-01T00:00:00Z",
    )

results = [
    _make_run_result("random",    seed=42,  best_score=10.0),
    _make_run_result("random",    seed=123, best_score=25.0),
    _make_run_result("random",    seed=456, best_score=18.0),
    _make_run_result("iterative", seed=42,  best_score=35.0),
    _make_run_result("iterative", seed=123, best_score=40.0),
]
print(f"Created {len(results)} RunResult objects")

In [ ]:
# Save to disk and load back
with tempfile.TemporaryDirectory() as tmpdir:
    saved_paths = [r.save(tmpdir) for r in results]
    print("Saved files:")
    for p in saved_paths:
        print(f"  {p.name}")

    # Load back the first result
    with open(saved_paths[0]) as fh:
        loaded = json.load(fh)
    print(f"\nLoaded back: best_score={loaded['best_score']}, seed={loaded['seed']}")

In [ ]:
# Aggregate statistics
agg = aggregate_results(results)
print("Aggregated statistics:")
for key, stats in sorted(agg.items()):
    scores_str = ", ".join(f"{s:.1f}" for s in stats["scores"])
    stdev_str = f", stdev={stats['stdev']:.2f}" if "stdev" in stats else ""
    print(f"  {key}")
    print(f"    scores=[{scores_str}]  mean={stats['mean']:.2f}  max={stats['max']:.2f}{stdev_str}")

In [ ]:
# Show RunResult.summary() for each result
print("Summaries:")
for r in results:
    s = r.summary()
    print(f"  {s['sampler_type']:10s} seed={s['seed']:3d}  best_score={s['best_score']}")

---
## 13. Runner Dry Run

In [ ]:
# Create a Runner from a config dict
runner_cfg = ExperimentConfig._from_dict({
    "vllm": {"base_url": "http://localhost:8000/v1", "api_key": "dummy"},
    "models": [{"name": "gpt-oss-120b"}, {"name": "qwen3.5-32b"}],
    "problems": ["combinatorics/cap_set"],
    "samplers": [
        {"type": "random", "num_samples": 50, "mode": "full_rewrite"},
        {"type": "iterative", "num_steps": 10, "samples_per_step": 5, "mode": "edit"},
    ],
    "runs": {"seeds": [42, 123, 456]},
    "output_dir": "experiments/results",
})

runner = Runner(runner_cfg)
print("Runner created ✓")

In [ ]:
# Dry run — prints what would execute without running anything
runner.dry_run()

In [ ]:
# Inspect _resolve_problems with explicit list
problems = runner._resolve_problems()
print("Resolved problems:", problems)

# Inspect _config_to_dict
config_dict = _config_to_dict(runner_cfg)
print("\nConfig dict keys:", list(config_dict.keys()))
print("JSON size:", len(json.dumps(config_dict)), "chars")

In [ ]:
# Test _build_sampler directly
from autoresearch_bench.config import SamplerConfig

mock_client = MagicMock(spec=LLMClient)
builder = PromptBuilder(mode="full_rewrite")

for s_type, cls_name in [("random", "RandomSampler"), ("iterative", "IterativeSampler")]:
    scfg = SamplerConfig(type=s_type)
    sampler = _build_sampler(
        sampler_cfg=scfg,
        client=mock_client,
        model="gpt-oss-120b",
        prompt_builder=builder,
        llm_params={},
        eval_max_workers=4,
    )
    print(f"  type={s_type!r:12s} → {type(sampler).__name__}")